# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 2 — Web Scraping Falcon 9 and Falcon Heavy Launch Records from Wikipedia

**Objective:** Scrape historical Falcon 9 launch records from the supplied Wikipedia snapshot using `requests` and `BeautifulSoup`, parse the HTML tables, create a Pandas DataFrame, and export the results to CSV.

## 1. Import required packages

In [ ]:
# Install these packages if they are not already installed:
# %pip install beautifulsoup4 requests pandas

import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

## 2. Define helper functions

In [ ]:
def date_time(table_cells):
    """Return date and time values from a launch table cell."""
    values = [value.strip() for value in table_cells.strings if value.strip()]
    return values[:2]


def booster_version(table_cells):
    """Extract the booster version from a launch table cell."""
    values = [
        value.strip()
        for index, value in enumerate(table_cells.strings)
        if index % 2 == 0 and value.strip()
    ]
    return ''.join(values[:-1]) if len(values) > 1 else ''.join(values)


def landing_status(table_cells):
    """Return the landing status from a launch table cell."""
    values = [value.strip() for value in table_cells.strings if value.strip()]
    return values[0] if values else ''


def get_mass(table_cells):
    """Extract the payload mass including its kg unit, or 0 if unavailable."""
    mass = unicodedata.normalize("NFKD", table_cells.get_text(" ", strip=True)).strip()
    if mass and "kg" in mass:
        return mass[:mass.find("kg") + 2]
    return 0


def extract_column_from_header(row):
    """Clean and return a table column heading."""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()

    column_name = ' '.join(str(content) for content in row.contents)
    if not column_name.strip().isdigit():
        return column_name.strip()
    return None

## 3. Set the Wikipedia snapshot URL and request headers

In [ ]:
static_url = (
    "https://en.wikipedia.org/w/index.php?"
    "title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/91.0.4472.124 Safari/537.36"
    )
}

## TASK 1: Request the Falcon 9 launch Wikipedia page

In [ ]:
# Send an HTTP GET request to the Wikipedia snapshot.
response = requests.get(static_url, headers=headers, timeout=30)
response.raise_for_status()

print("HTTP status code:", response.status_code)

In [ ]:
# Parse the HTML response using BeautifulSoup.
soup = BeautifulSoup(response.text, 'html.parser')

# Verify that the page was parsed correctly.
print(soup.title)

## TASK 2: Extract all column names from the HTML table header

In [ ]:
# Find all HTML tables on the page.
html_tables = soup.find_all('table')

print("Number of tables found:", len(html_tables))

# The third table (index 2) contains the launch records in this snapshot.
first_launch_table = html_tables[2]
print(first_launch_table)

In [ ]:
# Extract the column names from the table header.
column_names = []

for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)

## TASK 3: Create a DataFrame by parsing the launch HTML tables

In [ ]:
# Create a dictionary using the extracted column names.
launch_dict = dict.fromkeys(column_names)

# Remove the irrelevant date/time heading generated from the header.
launch_dict.pop('Date and time ( )', None)

# Initialize the columns that will store the extracted launch information.
required_columns = [
    'Flight No.', 'Launch site', 'Payload', 'Payload mass',
    'Orbit', 'Customer', 'Launch outcome',
    'Version Booster', 'Booster landing', 'Date', 'Time'
]

for column in required_columns:
    launch_dict[column] = []

print("Data columns prepared:", list(launch_dict.keys()))

### Parse each launch record

The code below walks through the Wikipedia launch tables. It checks that each row begins with a numeric flight number, then extracts the corresponding cell values.

In [ ]:
extracted_row = 0

for table_number, table in enumerate(
    soup.find_all('table', class_='wikitable plainrowheaders collapsible')
):
    for rows in table.find_all('tr'):
        # Identify rows whose first heading is a numeric flight number.
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        else:
            flag = False

        row = rows.find_all('td')

        if flag and len(row) >= 9:
            extracted_row += 1

            # Flight number
            launch_dict['Flight No.'].append(flight_number)

            # Date and time
            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',') if len(datatimelist) > 0 else ''
            time = datatimelist[1] if len(datatimelist) > 1 else ''
            launch_dict['Date'].append(date)
            launch_dict['Time'].append(time)

            # Booster version
            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.get_text(strip=True) if row[1].a else row[1].get_text(strip=True)
            launch_dict['Version Booster'].append(bv)

            # Launch site
            launch_site = row[2].a.get_text(strip=True) if row[2].a else row[2].get_text(' ', strip=True)
            launch_dict['Launch site'].append(launch_site)

            # Payload
            payload = row[3].a.get_text(strip=True) if row[3].a else row[3].get_text(' ', strip=True)
            launch_dict['Payload'].append(payload)

            # Payload mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            # Orbit
            orbit = row[5].a.get_text(strip=True) if row[5].a else row[5].get_text(' ', strip=True)
            launch_dict['Orbit'].append(orbit)

            # Customer
            customer = row[6].a.get_text(strip=True) if row[6].a else row[6].get_text(' ', strip=True)
            launch_dict['Customer'].append(customer)

            # Launch outcome
            launch_outcome_values = [x.strip() for x in row[7].strings if x.strip()]
            launch_outcome = launch_outcome_values[0] if launch_outcome_values else ''
            launch_dict['Launch outcome'].append(launch_outcome)

            # Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print("Total launch records extracted:", extracted_row)

## 4. Create and inspect the Pandas DataFrame

In [ ]:
# Convert the extracted dictionary into a Pandas DataFrame.
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})

print("DataFrame shape:", df.shape)
display(df.head())

In [ ]:
# Display DataFrame information and missing-value counts.
df.info()
display(df.isnull().sum())

## 5. Export the scraped data to CSV

In [ ]:
output_file = 'spacex_web_scraped.csv'
df.to_csv(output_file, index=False)

print(f"Saved {len(df)} launch records to {output_file}")

## Lab 2 completed

This notebook:
1. Requested the specified Wikipedia snapshot.
2. Parsed the HTML page using BeautifulSoup.
3. Located the launch-record tables.
4. Extracted table headings and launch information.
5. Created a Pandas DataFrame.
6. Exported the results to `spacex_web_scraped.csv`.

**Run the notebook from top to bottom with an internet connection.** Wikipedia's HTML structure can change over time; this lab uses the historical page snapshot specified in the assignment.